# Step 13.2 — Fused QKV Projection：为什么可以用一个 `Linear(C, 3C)` 一次得到 Q / K / V？

Step 11–12 里，为了把角色讲清楚，我们一直写成三套独立 projection：

$$
Q=XW_Q,
\qquad
K=XW_K,
\qquad
V=XW_V.
$$

到了真正的 Transformer / GPT 实现里，一个很常见的写法是把这三次 projection **打包成一次更大的 Linear**：

```text
X : (B,T,C)
↓
Linear(C, 3C)
↓
QKV_fused : (B,T,3C)
↓ split last axis into 3 equal chunks
Q : (B,T,C)
K : (B,T,C)
V : (B,T,C)
```

这叫 **fused QKV projection（融合 QKV 投影）**。

本节只解释这一层打包与拆分：一个 `Linear(C,3C)`、Q/K/V split、参数数量与 shape 的对应关系；不进入 Step 13.3 的 `C = H × D` head-channel split。


## 1. 三个 Projection Matrix 可以横向拼成一个大 Matrix

先用课程一直采用的 row-vector 数学记号。

原来：

$$
W_Q\in\mathbb R^{C\times C},
\qquad
W_K\in\mathbb R^{C\times C},
\qquad
W_V\in\mathbb R^{C\times C}.
$$

这里为了衔接 GPT-style full-width QKV，暂时让 Q/K/V projection 输出 width 都等于 model width `C`。

把三个矩阵沿 **output-feature direction** 横向拼接：

$$
\boxed{
W_{QKV}
=
[\,W_Q\mid W_K\mid W_V\,]
}
$$

于是：

$$
W_{QKV}
\in
\mathbb R^{C\times 3C}.
$$

现在只做一次：

$$
\boxed{
Y_{QKV}
=
XW_{QKV}
}
$$

如果：

$$
X:(B,T,C),
$$

那么：

$$
Y_{QKV}:(B,T,3C).
$$

因为 `W_QKV` 本身就是 `[W_Q | W_K | W_V]`，所以 fused output 的最后一维实际上就是：

```text
[ Q features | K features | V features ]
```

因此再把最后一个 axis 等分成三段，就恢复 Q、K、V。


## 2. 一个可以手算的 `C=2` 例子

取：

$$
X=
\begin{bmatrix}
1&2\\
0&1
\end{bmatrix}.
$$

三套原始 projection matrices：

$$
W_Q=
\begin{bmatrix}
1&0\\
0&1
\end{bmatrix},
\qquad
W_K=
\begin{bmatrix}
1&1\\
0&1
\end{bmatrix},
$$

$$
W_V=
\begin{bmatrix}
2&0\\
1&1
\end{bmatrix}.
$$

分别算：

$$
Q=XW_Q
=
\begin{bmatrix}
1&2\\
0&1
\end{bmatrix},
$$

$$
K=XW_K
=
\begin{bmatrix}
1&3\\
0&1
\end{bmatrix},
$$

$$
V=XW_V
=
\begin{bmatrix}
4&2\\
1&1
\end{bmatrix}.
$$

现在横向拼起来：

$$
W_{QKV}
=
\begin{bmatrix}
1&0&1&1&2&0\\
0&1&0&1&1&1
\end{bmatrix}
\in
\mathbb R^{2\times6}.
$$

一次矩阵乘法：

$$
XW_{QKV}
=
\begin{bmatrix}
1&2&1&3&4&2\\
0&1&0&1&1&1
\end{bmatrix}.
$$

按最后一维每 2 个 features 切开：

```text
[1,2 | 1,3 | 4,2]
[0,1 | 0,1 | 1,1]
```

恰好重新得到 Q、K、V。

所以 fused projection 不是新的 Attention 数学，只是把三次相同输入 X 的线性映射合并成一次更大的线性映射。


In [ ]:
import mlx.core as mx

X = mx.array([
    [1.0, 2.0],
    [0.0, 1.0],
])  # (T,C) = (2,2)

W_Q = mx.array([
    [1.0, 0.0],
    [0.0, 1.0],
])  # (C,C)

W_K = mx.array([
    [1.0, 1.0],
    [0.0, 1.0],
])  # (C,C)

W_V = mx.array([
    [2.0, 0.0],
    [1.0, 1.0],
])  # (C,C)

Q_separate = X @ W_Q
K_separate = X @ W_K
V_separate = X @ W_V

# Horizontal concatenation in the mathematical row-vector convention.
W_QKV = mx.concatenate([W_Q, W_K, W_V], axis=1)  # (C,3C)
QKV_fused = X @ W_QKV                            # (T,3C)

Q_fused, K_fused, V_fused = mx.split(
    QKV_fused,
    indices_or_sections=3,
    axis=-1,
)

print('W_QKV shape:', W_QKV.shape)
print('QKV fused shape:', QKV_fused.shape)
print('Q shape:', Q_fused.shape)
print('K shape:', K_fused.shape)
print('V shape:', V_fused.shape)

print('Q separate / fused:', Q_separate, Q_fused)
print('K separate / fused:', K_separate, K_fused)
print('V separate / fused:', V_separate, V_fused)

# This master notebook is unexecuted.
# The equality is a mathematical consequence of how W_QKV was constructed.


## 3. 映射到 MLX：`nn.Linear(C, 3C)`

项目固定 MLX 0.32.2。

该版本：

```python
nn.Linear(input_dims, output_dims, bias=True)
```

内部 stored weight shape 是：

$$
(output\_dims,input\_dims).
$$

所以代码：

```python
qkv_proj = nn.Linear(
    input_dims=C,
    output_dims=3 * C,
    bias=False,
)
```

其 stored weight：

$$
qkv\_proj.weight:(3C,C).
$$

而数学 row-vector notation 中的：

$$
W_{QKV}:(C,3C).
$$

两者仍然是 Step 11.2 已经见过的转置布局关系。

Forward：

```python
qkv = qkv_proj(x)
```

把：

$$
(B,T,C)
$$

映射到：

$$
\boxed{(B,T,3C)}.
$$


In [ ]:
import mlx.nn as nn

B, T, C = 2, 4, 6

qkv_proj = nn.Linear(
    input_dims=C,
    output_dims=3 * C,
    bias=False,
)

x = mx.zeros((B, T, C))
qkv = qkv_proj(x)  # (B,T,3C)

q, k, v = mx.split(
    qkv,
    indices_or_sections=3,
    axis=-1,
)

print('x:', x.shape)
print('stored fused weight:', qkv_proj.weight.shape)
print('qkv:', qkv.shape)
print('q:', q.shape)
print('k:', k.shape)
print('v:', v.shape)


## 4. `mx.split(..., 3, axis=-1)` 到底在做什么？

MLX 0.32.2 的 `mx.split` 可以写：

```python
mx.split(
    a,
    indices_or_sections=3,
    axis=-1,
)
```

当 `indices_or_sections` 是整数 3 时，它会要求指定 axis 可以被等分成 3 份。

这里：

$$
qkv:(B,T,3C).
$$

最后一个 axis 长度：

$$
3C.
$$

所以天然可以等分成：

$$
C+C+C.
$$

得到：

$$
Q:(B,T,C),
\qquad
K:(B,T,C),
\qquad
V:(B,T,C).
$$

这里的 split 只是把 fused feature axis 按约定切成三段。

本节不讨论底层是否发生物理 copy；那属于 implementation/runtime detail，不影响当前 tensor semantics。


## 5. Fused Projection 会减少参数数量吗？不会

这是本节最容易出现的误解。

先假设 `bias=False`。

### 三套独立 Linear

每个：

$$
W_Q,W_K,W_V:(C,C).
$$

每个都有：

$$
C^2
$$

个 weights。

总计：

$$
\boxed{3C^2}.
$$

### 一个 Fused Linear

数学矩阵：

$$
W_{QKV}:(C,3C).
$$

参数数量：

$$
C\times3C
=
\boxed{3C^2}.
$$

完全一样。

如果 `bias=True`：

- 三个独立 projections 各有 `C` 个 bias，总共 `3C`；
- fused projection 有一个长度 `3C` 的 bias；

总数仍然一样。

所以：

> **Fused QKV 是 parameter packing / computation organization，不是 parameter compression。**


## 6. 那为什么还要 Fuse？

概念上：

```text
Separate:
X -> q_proj
X -> k_proj
X -> v_proj

Fused:
X -> one larger qkv_proj
       ↓
      split
```

数学自由度没有减少。

工程上，一个较大的 projection 通常可以：

- 用一次更大的 matrix multiplication 产生全部 Q/K/V features；
- 减少 Python/module 调用与 kernel-dispatch 类开销；
- 让后续 tensor organization 更集中。

但不要把它说成：

> “任何设备、任何实现中一定严格快 3 倍。”

真实性能仍取决于 framework、compiler、hardware、shape 与 kernel implementation。

因此更准确的结论是：

> **Fused QKV 保持相同参数表达能力，同时把三次同源线性投影组织成一个更适合高效实现的大 projection。**


## 7. Step 13.2 材料边界

- 三套 full-width projections `W_Q/W_K/W_V:(C,C)` 可以沿 output-feature direction 拼成 `W_QKV:(C,3C)`。
- `X:(B,T,C)` 经过一个 fused projection 后得到 `(B,T,3C)`，最后一个 axis 的三段依次解释为 Q、K、V。
- MLX 0.32.2 `nn.Linear(C,3C)` stored weight shape 是 `(3C,C)`；数学 row-vector matrix 则写 `(C,3C)`。
- `mx.split(qkv, indices_or_sections=3, axis=-1)` 把 `(B,T,3C)` 等分成三个 `(B,T,C)` tensors。
- 三个独立 `Linear(C,C)` 与一个 `Linear(C,3C)` 在相同 bias 设定下参数数量相同：bias=False 时都是 `3C^2`；bias=True 时都是 `3C^2+3C`。
- fused projection 的优势是参数打包与计算组织，不是减少数学参数，也不保证固定倍数的运行加速。
- 本轮只准备 Step 13.2；不进入 Step 13.3，不介绍 `C=H×D`、`(B,T,C)->(B,T,H,D)`、head-channel split 或后续 transpose/concatenate。
